<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px 32px; border-radius: 16px; margin-bottom: 8px;">
  <div style="display: flex; align-items: center; gap: 16px; margin-bottom: 12px;">
    <img src="https://raw.githubusercontent.com/PositiveMatician/GuGa-Nexus/latest/app-stable/app/src/main/assets/logo.png" width="64" height="64" style="border-radius: 12px;" />
    <div>
      <h1 style="color: #e0e0ff; margin: 0; font-size: 2em; letter-spacing: -0.5px;">GuGa Nexus</h1>
      <p style="color: #8892b0; margin: 4px 0 0; font-size: 1em;">Bridge your Linux machine and Android device — no cloud, no subscription.</p>
    </div>
  </div>
  <div style="display: flex; gap: 8px; flex-wrap: wrap; margin-top: 16px;">
    <span style="background: #0f3460; color: #64ffda; padding: 4px 12px; border-radius: 20px; font-size: 0.78em; border: 1px solid #64ffda44;">🔒 AES-256-GCM</span>
    <span style="background: #0f3460; color: #ccd6f6; padding: 4px 12px; border-radius: 20px; font-size: 0.78em; border: 1px solid #ccd6f644;">🐧 Linux</span>
    <span style="background: #0f3460; color: #ccd6f6; padding: 4px 12px; border-radius: 20px; font-size: 0.78em; border: 1px solid #ccd6f644;">📱 Android</span>
    <span style="background: #0f3460; color: #ccd6f6; padding: 4px 12px; border-radius: 20px; font-size: 0.78em; border: 1px solid #ccd6f644;">☁️ Cloudflare Tunnel</span>
  </div>
</div>

---
## 🚀 Step 1 — Installation

Install the GuGa server package and the `guga` CLI tool.

In [ ]:
# Install GuGa Nexus server from GitHub (latest)
!pip install git+https://github.com/PositiveMatician/GuGa-Nexus.git@latest#subdirectory=server

# Install the guga CLI
!pip install guga

---
## ⚙️ Step 2 — Install Background Service

Sets up GuGa as a systemd service so it auto-starts on your machine. The `--choices "2,,2"` flag pre-fills the interactive prompts for non-interactive environments like Colab.

In [ ]:
!guga --install-service --choices "2,,2"

---
## 🌐 Step 3 — Start the Server

Start the GuGa backend. Use `--mode public` to expose the server via a **Cloudflare Tunnel** so your phone can connect from anywhere — not just your local network.

In [ ]:
# Foreground start (blocks the cell — use the background method below for long sessions)
!guga --start-server --mode public

### 🔁 Background Server (recommended for Colab)

Spawns the server as a detached process so Jupyter doesn't wait for it. Tails the log file to surface the public tunnel URL once it's ready.

In [ ]:
import subprocess
import time
import os

print("Spawning background server on port 6769 (public)...")

subprocess.Popen(
    ["guga", "--start-server", "--mode", "public", "-b"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    preexec_fn=os.setpgrp  # Detach from Colab's process group
)

log_path = os.path.expanduser("~/.guga/logs/server_6769.log")
print("Waiting for server to initialize...")

for _ in range(30):  # Wait up to 30 seconds
    time.sleep(1)
    if os.path.exists(log_path):
        with open(log_path, "r") as f:
            content = f.read()
            if "TUNNEL URL:" in content or "address →" in content:
                lines = content.splitlines()[-5:]
                print("\n" + "\n".join(lines))
                break
else:
    print("Timed out waiting for logs — the server may still be starting up.")

---
## 📊 Step 4 — Check Server Status

In [ ]:
!guga --status

---
## 📱 Step 5 — Pair Your Android Device

Display a QR code in the terminal. Open the **GuGa Android app** and scan it to pair.

In [ ]:
!guga --qr

---
## ✅ Step 6 — Approve Paired Devices

Approve all pending device pairing requests non-interactively. Run this right after scanning the QR code.

In [ ]:
!guga --approve -A

  ✓ Approved 2 devices.


---
## 💡 Usage Examples

The sections below demonstrate the core GuGa features you can drop into your own notebooks.

### 1. 🔔 Send a Notification

Push a message straight to your paired phone. Perfect for alerting yourself when a long-running cell (model training, data processing) finishes.

In [ ]:
import time

# Simulate a long-running task
time.sleep(2)

# Send notification on completion
!guga "The cell has finished executing!"

### 2. ❓ Interactive Prompts — Ask the User

Pause execution, send a question to your phone, and block until you reply. The response is captured and returned to your notebook.

In [ ]:
reply = !guga --ask-user "Should we save the model checkpoint now? (yes/no)"
print(f"User replied: {reply[0]}")

### 3. 🔁 Wrap Long Commands

Prefix any shell command with `guga run`. When it finishes, your phone receives the exit status (success or failure) automatically.

In [ ]:
!guga -r "sleep 3"

### 4. 🐍 Use GuGa as a Python Module

Import GuGa's internal functions directly for native Python integration — no shell commands needed.

> **Note:** Some CLI functions call `sys.exit()` on failure. Wrapping calls in `try/except SystemExit` prevents your Jupyter kernel from restarting on network errors.

In [ ]:
from guga.cli import broadcast_message, guga_ask_user

# Send a notification directly from Python
broadcast_message(
    message="Training completed successfully!",
    port=6769,
    silent=False,
    title="Model Update"
)

In [ ]:
from guga.cli import broadcast_message, guga_ask_user

DEVICE_ID = ""#@param{type:"string"}
# Ask the user a question and wait for their reply
print("Waiting for user response...")
try:
    reply = guga_ask_user(
        prompt="Upload model to HuggingFace? (y/n)",
        port=6769,
        device_id="",
        timeout=60,
        quiet=True
    )
    print(f"Python received: {reply}")
except SystemExit:
    print("Request timed out or no device was connected.")

### 5. 🧪 Quick Test — Send a "Hey"

Verify end-to-end connectivity with a simple test message.

In [ ]:
!guga "hey"

---
## 🛑 Stop the Server

Stop all running GuGa background servers when you're done. The `--choices "A"` flag skips the interactive confirmation.

In [ ]:
!guga --stop-server --choices "A"